# Suivre la dérivée à travers deux poids

Ouvrir la remontée d’autograd quand une succession de couches devient difficile à expliquer.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#retropropagation-a-la-main).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


La boucle fonctionne, mais une question peut rester : comment un poids placé au début du modèle reçoit-il sa dérivée ? Gardons seulement deux multiplications, sans classification.

Les nombres sont choisis pour le calcul : entrée x=2, poids w1=3 et w2=−1. La sortie visée est zéro. Ce n’est pas un nouvel extrait audio.

La passe directe donne h=6, puis y=−6 et une perte de 36. Retrouvons maintenant l’influence de chaque opération en sens inverse.

| Dérivée | Calcul local puis trajet | Valeur |
| --- | --- | --- |
| dL/dy | 2y | −12 |
| dL/dw2 | (dL/dy) × h | −72 |
| dL/dh | (dL/dy) × w2 | 12 |
| dL/dw1 | (dL/dh) × x | 24 |

Dans un produit, dériver par rapport à un facteur donne l’autre facteur. Le gradient reçu est ensuite multiplié par cette influence locale.

Les appels retain_grad ne changent aucun nombre et ne calculent pas encore de dérivée. Ils demandent de garder y.grad et h.grad après backward. Les poids créés directement avec requires_grad=True n’en ont pas besoin.

Retirer h.retain_grad() empêcherait de lire h.grad dans cet exemple ; le gradient de w1 serait pourtant toujours calculé. Garder un résultat pour inspection et propager une influence sont deux opérations distinctes.

## Démonstration 1 · Comparer la remontée manuelle à autograd

Inspecter deux poids et deux intermédiaires sans modifier leurs valeurs.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Calculer l’état suivant

Seuls w1 et w2 sont des valeurs ajustables.

In [4]:
x = torch.tensor(2.)
w1 = torch.tensor(3., requires_grad=True)
w2 = torch.tensor(-1., requires_grad=True)


In [5]:
observer('x', x)
observer('w1', w1)
observer('w2', w2)

x forme () type torch.float32 requires_grad False
  valeurs : 2.0
w1 forme () type torch.float32 requires_grad True
  valeurs : 3.0
w2 forme () type torch.float32 requires_grad True
  valeurs : -1.0


### Calculer l’état suivant

La multiplication crée h. retain_grad prépare la lecture ultérieure de h.grad.

Cette ligne garde une dérivée intermédiaire après backward ; elle ne déclenche pas la remontée.

In [6]:
h = w1 * x
h.retain_grad()


In [7]:
observer('h', h)

h forme () type torch.float32 requires_grad True
  valeurs : 6.0


### Calculer l’état suivant

La cible zéro permet d’écrire directement le carré de y.

In [8]:
y = w2 * h
y.retain_grad()
perte = y.square()


In [9]:
observer('y', y)
observer('perte', perte)

y forme () type torch.float32 requires_grad True
  valeurs : -6.0
perte forme () type torch.float32 requires_grad True
  valeurs : 36.0


### Calculer les gradients

La règle de la chaîne combine les facteurs du tableau.

In [10]:
perte.backward()


In [11]:
observer_entrainement({'x': x, 'w1': w1, 'w2': w2, 'h': h, 'y': y, 'perte': perte}, montrer_poids=False)

w1.grad forme () type torch.float32 requires_grad False
  valeurs : 24.0
w2.grad forme () type torch.float32 requires_grad False
  valeurs : -72.0


### Lire les sorties

item extrait les scalaires pour les afficher. Aucun optimiseur n’est appelé.

In [12]:
print('h, y, perte :', h.item(), y.item(), perte.item())
print('gradients y, h :', y.grad.item(), h.grad.item())
print('gradients w1, w2 :', w1.grad.item(), w2.grad.item())

h, y, perte : 6.0 -6.0 36.0
gradients y, h : -12.0 12.0
gradients w1, w2 : 24.0 -72.0


In [13]:
assert bool(h.item() == 6 and y.item() == -6 and perte.item() == 36), 'retropropagation-a-la-main/main: h.item() == 6 and y.item() == -6 and perte.item() == 36'
assert bool(w1.grad.item() == 24 and w2.grad.item() == -72), 'retropropagation-a-la-main/main: w1.grad.item() == 24 and w2.grad.item() == -72'
assert bool(h.grad.item() == 12 and y.grad.item() == -12), 'retropropagation-a-la-main/main: h.grad.item() == 12 and y.grad.item() == -12'
assert bool(w1.item() == 3 and w2.item() == -1), 'retropropagation-a-la-main/main: w1.item() == 3 and w2.item() == -1'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Les gradients de poids existent indépendamment de la conservation des gradients intermédiaires.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Complétez le gradient du premier poids dans le graphe x=2, w1=3, w2=−1.

In [14]:
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w1.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Remontez −12 vers h avec le facteur −1, puis vers w1 avec le facteur 2.

</details>

### Exercice 2

Complétez cette fois le gradient du second poids, avec le même calcul.

In [15]:
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = ...  # Remplacer les points par votre réponse.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w2.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

La variation locale de y par rapport à w2 vaut h=6. Multipliez-la par −12.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = 24.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w1.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_w1 = torch.tensor(3.,requires_grad=True)
exo_w2 = torch.tensor(-1.,requires_grad=True)
exo_h = exo_w1*2
exo_y = exo_w2*exo_h
(exo_y**2).backward()
exo_gradient = -72.
if exo_gradient is Ellipsis or exo_gradient is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_gradient)-exo_w2.grad.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()